# Pilot P3 + P4: FQ-QAT Stability and QUAIL Baseline

## IMPORTANT: Run on SINGLE T4 GPU (not dual)

- **P3**: Tests whether FQ-QAT + NPO training is stable on forget01
- **P4**: Establishes the Q-INT4 score for the QUAIL baseline
- **Estimated total time**: 60-90 minutes
- **Results saved to**: /kaggle/working/results/

### Before running:
1. Set accelerator to **GPU T4 x1** (single GPU)
2. Ensure **Internet** is toggled ON
3. Replace `YOUR_HF_TOKEN` with your actual HuggingFace token

## Cell 1: Setup and Dependencies (Isolated VENV)

In [ ]:
import os
import subprocess
import sys

# ── Configuration ────────────────────────────────────────────
HF_TOKEN = "YOUR_HF_TOKEN"  # <-- REPLACE THIS
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["TRANSFORMERS_NO_FLASH_ATTN"] = "1"
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("=== Cloning Repositories ===")

# Clone fq-ssiuu
if not os.path.exists("/kaggle/working/fq-ssiuu"):
    subprocess.run(
        "git clone --branch pilot/p3-p4 https://github.com/ibn-e-Muhammad/fq-ssiuu.git /kaggle/working/fq-ssiuu",
        shell=True, check=True,
    )
else:
    subprocess.run(
        "cd /kaggle/working/fq-ssiuu && git fetch origin && git checkout pilot/p3-p4 && git pull origin pilot/p3-p4",
        shell=True, check=True,
    )

# Clone OpenUnlearning
if not os.path.exists("/kaggle/working/open-unlearning"):
    subprocess.run(
        "git clone --branch fq-ssiuu-integration https://github.com/ibn-e-Muhammad/open-unlearning.git /kaggle/working/open-unlearning",
        shell=True, check=True,
    )
else:
    subprocess.run(
        "cd /kaggle/working/open-unlearning && git fetch origin && git checkout fq-ssiuu-integration",
        shell=True, check=True,
    )

print("\n=== Patching Dependencies ===")
subprocess.run("sed -i '/torch==/d' /kaggle/working/open-unlearning/requirements.txt", shell=True)
subprocess.run("sed -i '/torchvision==/d' /kaggle/working/open-unlearning/requirements.txt", shell=True)
subprocess.run("sed -i '/bitsandbytes==/d' /kaggle/working/open-unlearning/requirements.txt", shell=True)
subprocess.run("sed -i 's/hf_vlms/huggingface/g' /kaggle/working/open-unlearning/src/evals/lm_eval.py", shell=True)

print("\n=== Creating Isolated Virtual Environment ===")
# We completely abandon Kaggle's broken system environment to avoid all Numpy/Scipy conflicts.
# Use --without-pip because Kaggle's ensurepip is broken.
subprocess.run(f"{sys.executable} -m venv --without-pip /kaggle/working/venv", shell=True, check=True)

VENV_PIP = "/kaggle/working/venv/bin/pip"
VENV_PY = "/kaggle/working/venv/bin/python"

# Manually install pip to bypass ensurepip bugs
subprocess.run(f"curl -sS https://bootstrap.pypa.io/get-pip.py | {VENV_PY}", shell=True, check=True)

print("\n=== Installing Clean Dependencies into VENV ===")
# Install wrapt first to silence Kaggle's annoying sitecustomize.py warnings
subprocess.run(f"{VENV_PIP} install wrapt -q", shell=True)

# REMOVED the -q flag so you can actually see the progress bar!
# Torch is ~2.5 GB, it takes ~2-3 minutes to download. It is NOT frozen, just downloading!
print("\n[INFO] Downloading PyTorch (this is ~2.5 GB and takes 2-3 minutes)...\n")
subprocess.run(f"{VENV_PIP} install torch torchvision torchaudio", shell=True)

print("\n[INFO] Installing strict math matrix...\n")
subprocess.run(f"{VENV_PIP} install numpy==1.26.4 scipy==1.13.1 scikit-learn==1.4.2 peft lm-eval==0.4.4 bitsandbytes==0.45.5 captum", shell=True)

print("\n[INFO] Installing local packages...\n")
subprocess.run(f"{VENV_PIP} install -e /kaggle/working/open-unlearning", shell=True)
subprocess.run(f"{VENV_PIP} install -e /kaggle/working/fq-ssiuu --no-deps", shell=True)

print("\n=== Downloading Retain Logs ===")
subprocess.run(
    f"cd /kaggle/working/open-unlearning && {VENV_PY} setup_data.py --eval_logs",
    shell=True,
)

retain_logs = "/kaggle/working/open-unlearning/saves/eval/tofu_Llama-3.2-1B-Instruct_retain90/TOFU_EVAL.json"
if not os.path.exists(retain_logs):
    print("setup_data.py didn't produce the file. Trying direct download...")
    subprocess.run(f"{VENV_PIP} install huggingface_hub -q", shell=True)
    dl_script = r"""
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='open-unlearning/eval',
    allow_patterns='*.json',
    repo_type='dataset',
    local_dir='/kaggle/working/open-unlearning/saves/eval'
)
"""
    with open("/kaggle/working/dl.py", "w") as f:
        f.write(dl_script)
    subprocess.run(f"{VENV_PY} /kaggle/working/dl.py", shell=True)

assert os.path.exists(retain_logs), f"FATAL: Retain logs not found at {retain_logs}"
print(f"✅ Retain logs ready: {retain_logs}")
print("\n=== Setup Complete ===")

## Cell 2: Verify Setup (Inside VENV)

In [ ]:
import subprocess
import os

# Fix matplotlib crash caused by Kaggle's Jupyter environment bleeding into our VENV
os.environ["MPLBACKEND"] = "Agg"

# Force bitsandbytes to use CUDA 12.1 binaries if Kaggle host is on CUDA 13.0
os.environ["BNB_CUDA_VERSION"] = "121"

verify_script = r"""
import torch
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
import bitsandbytes
print(f"bitsandbytes: {bitsandbytes.__version__}")
import peft
print(f"peft: {peft.__version__}")
import captum
print(f"captum: {captum.__version__}")
print("\n✅ All checks passed inside isolated VENV.")
"""

with open("/kaggle/working/verify.py", "w") as f:
    f.write(verify_script)

subprocess.run("/kaggle/working/venv/bin/python /kaggle/working/verify.py", shell=True, text=True, env=os.environ)

## Cell 3: Run Pilot P3 (Inside VENV)

In [ ]:
import os, subprocess

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["TRANSFORMERS_NO_FLASH_ATTN"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONPATH"] = "/kaggle/working/fq-ssiuu/src"
os.environ["MPLBACKEND"] = "Agg"  # Fix matplotlib error
os.environ["BNB_CUDA_VERSION"] = "121"  # Fix bitsandbytes CUDA 13.0 error

OU_PATH = "/kaggle/working/open-unlearning"
OUTPUT_DIR = "/kaggle/working/results/pilot_p3"
VENV_PY = "/kaggle/working/venv/bin/python"

print("=== PILOT P3: FQ-QAT + NPO STABILITY TEST ===\n")

result = subprocess.run(
    [
        VENV_PY,
        "/kaggle/working/fq-ssiuu/experiments/pilot_p3_fqat_stability.py",
        "--open_unlearning_path", OU_PATH,
        "--drive_output", OUTPUT_DIR,
    ],
    capture_output=False,
    text=True,
    env=os.environ,
)

print(f"\n=== P3 Exit Code: {result.returncode} ===")

## Cell 4: Run Pilot P4 (Inside VENV)

In [ ]:
import os, subprocess

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["TRANSFORMERS_NO_FLASH_ATTN"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONPATH"] = "/kaggle/working/fq-ssiuu/src"
os.environ["MPLBACKEND"] = "Agg"  # Fix matplotlib error
os.environ["BNB_CUDA_VERSION"] = "121"  # Fix bitsandbytes CUDA 13.0 error

OU_PATH = "/kaggle/working/open-unlearning"
OUTPUT_DIR = "/kaggle/working/results/pilot_p4"
VENV_PY = "/kaggle/working/venv/bin/python"

print("=== PILOT P4: QUAIL BASELINE ===\n")

result = subprocess.run(
    [
        VENV_PY,
        "/kaggle/working/fq-ssiuu/experiments/pilot_p4_quail_baseline.py",
        "--open_unlearning_path", OU_PATH,
        "--drive_output", OUTPUT_DIR,
    ],
    capture_output=False,
    text=True,
    env=os.environ,
)

print(f"\n=== P4 Exit Code: {result.returncode} ===")